# Face attributes: print all extracted fields

For each detected face, print everything the pipeline extracts:

- **Detection:** bbox, size, confidence
- **Demographics:** age, gender (InsightFace genderage model)
- **Head pose:** pitch, yaw, roll
- **Landmarks:** 5-point kps, 106-point 2D, 68-point 3D (when present)
- **Clustering:** HDBSCAN label from `.scar` sidecar (if scanned)

**Emotion / mood** is not available — the `buffalo_l` model pack has no emotion head.

> **Sidecar:** `face.scrfd.faces` stores full attributes (age, gender, pose, landmarks). Load from `.scar` by default; set `FORCE_DETECT=True` to re-run SCRFD.

In [ ]:
import sys
from pathlib import Path

NOTEBOOKS_DIR = Path.cwd()
if NOTEBOOKS_DIR.name == "notebooks":
    REPO_ROOT = NOTEBOOKS_DIR.parent
else:
    REPO_ROOT = NOTEBOOKS_DIR
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from meta_face.imaging import load_image
from meta_face.tools.face_record import resolve_face_records

import _face_info

## User input

Set `IMAGE_PATH`. `FORCE_DETECT=False` loads full attributes from the sibling `.scar` (run `mf scan` first).

In [ ]:
IMAGE_PATH = Path("../test_images/20110903_172733.840.jpg")  # team photo, 17 faces
FORCE_DETECT = False  # True to re-run SCRFD instead of loading face.scrfd.* from .scar

if not IMAGE_PATH.is_file():
    raise FileNotFoundError(f"Set IMAGE_PATH to an existing image: {IMAGE_PATH}")

In [ ]:
image = load_image(IMAGE_PATH)
records, source = resolve_face_records(IMAGE_PATH, force=FORCE_DETECT, image=image)
cluster_labels = _face_info.cluster_labels_for_media(IMAGE_PATH)

verb = "Detected" if source == "detect" else "Loaded"
print(f"{verb} {len(records)} face(s) from {source} in {IMAGE_PATH.name}")

In [ ]:
_face_info.print_all_face_attributes(
    records,
    cluster_labels=cluster_labels,
    source=source,
)

In [ ]:
import json

# Compact JSON view (landmark arrays truncated for readability)
def slim_record(record: dict) -> dict:
    out = {}
    for key, value in record.items():
        if isinstance(key, str) and key.startswith("landmark_") and isinstance(value, list):
            out[key] = f"<{len(value)} points>"
        elif key == "kps" and isinstance(value, list):
            out[key] = value
        else:
            out[key] = value
    return out

print(json.dumps([slim_record(r) for r in records], indent=2))